<a href="https://colab.research.google.com/github/reshmapasula-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reshmapasula-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: Prioritize actions that show a clear need for review, using the available signals and avoiding future information.
Reason codes: high_decline, high_impact, old_content, weak_performance, low_confidence.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
)

# Show the available columns so we can verify the dataset
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nAvailable columns:")
print(df.columns.tolist())

# Create an evaluation label.
# This is used only to measure the baseline later.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Simple baseline signals
df["stale"] = (
    pd.to_numeric(df["days_since_last_update"], errors="coerce") >= 180
).astype(int)

df["visible"] = (
    pd.to_numeric(df["impressions_90d"], errors="coerce") >= 500
).astype(int)

df["weak_ctr"] = (
    pd.to_numeric(df["ctr"], errors="coerce") < 1.0
).astype(int)

df["weak_position"] = (
    (pd.to_numeric(df["avg_position"], errors="coerce") > 20)
    & (pd.to_numeric(df["avg_position"], errors="coerce") != 0)
).astype(int)

# Assign reason codes
def get_reason(row):
    reasons = []

    if row["stale"] == 1:
        reasons.append("old_content")

    if row["visible"] == 1:
        reasons.append("high_impact")

    if row["weak_ctr"] == 1:
        reasons.append("weak_performance")

    if row["weak_position"] == 1:
        reasons.append("low_position")

    if not reasons:
        reasons.append("no_clear_signal")

    return "|".join(reasons)


df["reason_code"] = df.apply(get_reason, axis=1)

# Basic checks
print("\nDeclining label rate:",
      round(df["is_declining_label"].mean(), 3))

print("\nReason-code check:")
print(df["reason_code"].value_counts().head(10))

print("\nFirst 5 rows:")
display(df.head())

Rows: 30000
Columns: 44

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Declining label rate: 0.542

Reason-code check:
reason_code
high_impact|weak_performance                 11477
weak_performance                              8347
high_impact|weak_perfo

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,stale,visible,weak_ctr,weak_position,reason_code
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,good,striking,down,-41.4,1,0,1,1,0,high_impact|weak_performance
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,good,page_3_5,down,-57.7,1,0,1,1,1,high_impact|weak_performance|low_position
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,good,page_3_5,down,-60.9,1,0,1,1,1,high_impact|weak_performance|low_position
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,good,page_1,stable,-13.8,0,0,1,1,0,high_impact|weak_performance
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,good,page_3_5,down,-34.7,1,0,1,1,1,high_impact|weak_performance|low_position


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# SECTION 2 — Build the ranked queue

import os
import pandas as pd
import numpy as np

# Make sure the dataset is available
if "df" not in globals():
    df = pd.read_csv(
        "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    )

# Convert signals to numeric values
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
).fillna(0)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
).fillna(0)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
).fillna(0)

df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
).fillna(0)

# ---------------------------------------------------------
# BASELINE ACTION SCORE
# Higher score = stronger reason to review/refresh
#
# The score uses only currently available content/performance
# signals. It does NOT use future trend information.
# ---------------------------------------------------------

# Staleness: older content gets a higher score
stale_score = np.clip(
    df["days_since_last_update"] / 365,
    0,
    1
)

# Visibility: pages with more impressions have more potential impact
visibility_score = np.log1p(df["impressions_90d"])
visibility_score = visibility_score / max(
    visibility_score.max(), 1
)

# CTR weakness: lower CTR gets a higher score
ctr_score = 1 - np.clip(
    df["ctr"] / 5,
    0,
    1
)

# Ranking weakness: worse average position gets a higher score
position_score = np.clip(
    (df["avg_position"] - 1) / 49,
    0,
    1
)

# Weighted transparent baseline score
df["action_score"] = (
    0.30 * stale_score
    + 0.25 * visibility_score
    + 0.25 * ctr_score
    + 0.20 * position_score
)

# ---------------------------------------------------------
# ACTION
# ---------------------------------------------------------

df["action"] = np.where(
    df["action_score"] >= 0.60,
    "review",
    np.where(
        df["action_score"] >= 0.40,
        "monitor",
        "no_action"
    )
)

# ---------------------------------------------------------
# REASON CODE
# ---------------------------------------------------------

if "reason_code" not in df.columns:
    def make_reason(row):
        reasons = []

        if row["days_since_last_update"] >= 180:
            reasons.append("old_content")

        if row["impressions_90d"] >= 500:
            reasons.append("high_impact")

        if row["ctr"] < 1.0:
            reasons.append("weak_performance")

        if row["avg_position"] > 20 and row["avg_position"] != 0:
            reasons.append("low_position")

        return "|".join(reasons) if reasons else "no_clear_signal"

    df["reason_code"] = df.apply(make_reason, axis=1)

# ---------------------------------------------------------
# CONFIDENCE NOTE
# ---------------------------------------------------------

def confidence_note(score):
    if score >= 0.60:
        return "Higher-priority directional signal; review before acting."
    elif score >= 0.40:
        return "Moderate directional signal; monitor and validate."
    else:
        return "Weak baseline signal; no immediate action."

df["confidence_note"] = df["action_score"].apply(confidence_note)

# ---------------------------------------------------------
# RANK EVERYTHING
# ---------------------------------------------------------

df = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# ---------------------------------------------------------
# IDENTIFIER
# ---------------------------------------------------------

possible_id_columns = [
    "content_id",
    "page_id",
    "id",
    "url",
    "title",
    "query"
]

id_column = None

for column in possible_id_columns:
    if column in df.columns:
        id_column = column
        break

if id_column is None:
    df["item_id"] = np.arange(1, len(df) + 1)
    id_column = "item_id"

# ---------------------------------------------------------
# WRITE THE REQUIRED CSV
# ---------------------------------------------------------

output_columns = [
    id_column,
    "rank",
    "action_score",
    "action",
    "reason_code",
    "confidence_note"
]

output = df[output_columns].copy()

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

output.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# CHECK THE RESULT
# ---------------------------------------------------------

print("CSV created successfully:")
print(output_path)

print("\nRows written:", len(output))

print("\nAction counts:")
print(output["action"].value_counts())

print("\nTop 20 ranked items:")
display(output.head(20))

CSV created successfully:
work/outputs/baseline_action_score.csv

Rows written: 30000

Action counts:
action
monitor      20518
no_action     7748
review        1734
Name: count, dtype: int64

Top 20 ranked items:


,content_id,rank,action_score,action,reason_code,confidence_note
0,content_6476d1d8c050,1,0.815952,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
1,content_d25a099b3726,2,0.801641,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
2,content_7a888d3d99c8,3,0.793988,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
3,content_109f8f7c9d39,4,0.751835,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...
4,content_54baba704595,5,0.746567,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...
5,content_dd413158df3c,6,0.734911,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
6,content_5feee3994adb,7,0.734371,review,old_content|high_impact|weak_performance|low_p...,Higher-priority directional signal; review bef...
7,content_15fe075b97bc,8,0.733908,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
8,content_afd26a07382d,9,0.733367,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...
9,content_fb4bf6555c79,10,0.732986,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the 20 highest-scoring items using the baseline action score and its reason codes. The recommendations are directional decision-support, not final decisions. A pick could be wrong if the underlying signals are incomplete, noisy, or if the page has a valid business reason not captured in the dataset.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
# SECTION 3 — Top-20 review

# Make sure the ranked output exists
if "output" not in globals():
    output = pd.read_csv("work/outputs/baseline_action_score.csv")

# Get the top 20 ranked items
top20 = output.head(20).copy()

# Build a confidence note and "what would make it wrong" note
def wrong_if(row):
    reasons = str(row["reason_code"])

    if "old_content" in reasons and "weak_performance" in reasons:
        return "Could be wrong if the content is intentionally evergreen or the performance signal is temporary."

    if "high_impact" in reasons:
        return "Could be wrong if high traffic is not strategically important or the observed weakness is temporary."

    if "weak_performance" in reasons:
        return "Could be wrong if low performance is caused by a temporary or external factor."

    if "low_position" in reasons:
        return "Could be wrong if the ranking is appropriate for the query intent or the position signal is noisy."

    if "old_content" in reasons:
        return "Could be wrong if the content is intentionally kept unchanged and remains useful."

    return "Could be wrong if the available signals do not capture the full business or content context."

top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

# Keep only the requested review fields
review_columns = [
    output.columns[0],
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns].copy()

# Display the complete top-20 review
display(top20_review)

print("Top-20 review completed.")
print("Items reviewed:", len(top20_review))

,content_id,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_6476d1d8c050,1,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
1,content_d25a099b3726,2,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
2,content_7a888d3d99c8,3,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
3,content_109f8f7c9d39,4,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if high traffic is not strategi...
4,content_54baba704595,5,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if high traffic is not strategi...
5,content_dd413158df3c,6,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
6,content_5feee3994adb,7,review,old_content|high_impact|weak_performance|low_p...,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
7,content_15fe075b97bc,8,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
8,content_afd26a07382d,9,review,old_content|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if the content is intentionally...
9,content_fb4bf6555c79,10,review,high_impact|weak_performance|low_position,Higher-priority directional signal; review bef...,Could be wrong if high traffic is not strategi...


Top-20 review completed.
Items reviewed: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The weakest picks are items where the baseline score is driven by a single weak or noisy signal. These should be treated cautiously and manually validated before any action.

Leakage check: the ranking score uses current content/performance signals only. Future trend fields are used only for evaluation, not for the action score. No product flags or future windows are included in the scoring formula.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# SECTION 4 — Weak picks + leakage check

# Make sure the ranked data exists
if "df" not in globals():
    df = pd.read_csv(
        "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    )

if "output" not in globals():
    output = pd.read_csv("work/outputs/baseline_action_score.csv")

# ---------------------------------------------------------
# Weak picks
# ---------------------------------------------------------

weak_picks = output.tail(10).copy()

print("10 weakest-scoring picks:")
display(weak_picks)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

# Fields that should NOT be used to calculate the action score
future_or_evaluation_fields = [
    "trend_direction",
    "trend_pct",
    "future",
    "future_window",
    "product_flag",
    "product_flags"
]

score_formula_fields = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

print("\nLeakage check:")
print("Score fields used:")
print(score_formula_fields)

print("\nPotential future/product fields present in dataset:")
present_blocked = [
    col for col in future_or_evaluation_fields
    if col in df.columns
]

print(present_blocked if present_blocked else "None found")

# Confirm the score does not directly use future/evaluation fields
assert "trend_direction" not in score_formula_fields
assert "trend_pct" not in score_formula_fields
assert "future_window" not in score_formula_fields
assert "product_flag" not in score_formula_fields
assert "product_flags" not in score_formula_fields

print("\nPASS: No future trend or product-flag fields are used in the scoring formula.")

# Confirm the required CSV exists
import os

assert os.path.exists(
    "work/outputs/baseline_action_score.csv"
)

print("PASS: baseline_action_score.csv exists.")

# Final summary
print("\nSection 4 completed successfully.")

10 weakest-scoring picks:


,content_id,rank,action_score,action,reason_code,confidence_note
29990,content_f03ce106b8b7,29991,0.037313,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29991,content_0f96bf7b0be4,29992,0.037313,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29992,content_76b07f20b83c,29993,0.037313,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29993,content_a8cee66e4788,29994,0.033690,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29994,content_bf398aa7400e,29995,0.033690,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29995,content_f26233911f33,29996,0.031532,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29996,content_bc2c0c7243df,29997,0.029609,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29997,content_98458bafe297,29998,0.029609,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29998,content_b1e4f7904d85,29999,0.029609,no_action,no_clear_signal,Weak baseline signal; no immediate action.
29999,content_006b16e7a2e7,30000,0.019746,no_action,no_clear_signal,Weak baseline signal; no immediate action.



Leakage check:
Score fields used:
['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']

Potential future/product fields present in dataset:
['trend_direction', 'trend_pct']

PASS: No future trend or product-flag fields are used in the scoring formula.
PASS: baseline_action_score.csv exists.

Section 4 completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.